In [1]:
%load_ext autoreload

In [2]:

import pandas as pd
import json
import sklearn
import glob
import pickle
from sklearn.model_selection import train_test_split
from collections import Counter


pd.set_option('display.width', None)
pd.set_option('display.max_colwidth', None)
pd.set_option('display.max_rows', None)
pd.set_option('display.max_columns', None)

In [3]:
%autoreload
import sys
sys.path.insert(0, '../../style_generation_pipeline')

from data import *
from cluster_representation import *

/mnt/swordfish-pool2/milad/conda-envs/gpu-env/lib/python3.11/site-packages/sentence_transformers/cross_encoder/CrossEncoder.py:11: TqdmExperimentalWarning: Using `tqdm.autonotebook.tqdm` in notebook mode. Use `tqdm.tqdm` instead to force console mode (e.g. in jupyter console)
  from tqdm.autonotebook import tqdm, trange
2025-02-11 06:43:33.874624: E external/local_xla/xla/stream_executor/cuda/cuda_fft.cc:477] Unable to register cuFFT factory: Attempting to register factory for plugin cuFFT when one has already been registered
E0000 00:00:1739274214.186535   12609 cuda_dnn.cc:8310] Unable to register cuDNN factory: Attempting to register factory for plugin cuDNN when one has already been registered
E0000 00:00:1739274214.286613   12609 cuda_blas.cc:1418] Unable to register cuBLAS factory: Attempting to register factory for plugin cuBLAS when one has already been registered
2025-02-11 06:43:34.929506: I tensorflow/core/platform/cpu_feature_guard.cc:210] This TensorFlow binary is optimize

In [4]:
path='/mnt/swordfish-pool2/milad/hiatus-data/explainability_all_data/'

### Merging data from phase 1 and 2 that was used to generate style features

In [96]:
phase1_docs = pd.read_json(path_or_buf='/mnt/swordfish-pool2/milad/hiatus-data/phase_1/explainability/training_candidates_and_queries.jsonl', lines=True)
phase2_docs = pd.read_json(path_or_buf='/mnt/swordfish-pool2/milad/hiatus-data/phase_2/explainability/all_documents_in_cross_genre.jsonl', lines=True)

In [97]:
all_docs = pd.concat([phase1_docs, phase2_docs]).reset_index()

In [101]:
all_docs[['authorID', 'fullText', 'documentID']].to_json(path + '/all_document.jsonl', orient='records', lines=True)

### Merging data from phase 1 and 2 to perform clustering

In [10]:
phase2_training_data = pd.read_json('/mnt/swordfish-pool2/milad/hiatus-data/phase_2/explainability/train_authors.json')
phase2_test_data     = pd.read_json('/mnt/swordfish-pool2/milad/hiatus-data/phase_2/explainability/test_authors.json')

phase1_training_data = pd.read_json('/mnt/swordfish-pool2/milad/hiatus-data/phase_1/training_authors.json')
phase1_test_data     = pd.read_json('/mnt/swordfish-pool2/milad/hiatus-data/phase_1/valid_authors.json')

In [17]:
training_authors = pd.concat([phase1_training_data, phase2_training_data]).reset_index()
test_authors = pd.concat([phase1_test_data, phase2_test_data]).reset_index()

In [18]:
training_authors.columns

Index(['level_0', 'index', 'authorID', 'fullText', 'documentID', 'source'], dtype='object')

In [19]:
print(phase1_training_data.authorID.nunique(), phase2_training_data.authorID.nunique(), training_authors.authorID.nunique())
print(phase1_test_data.authorID.nunique(), phase2_test_data.authorID.nunique(), test_authors.authorID.nunique())

4142 1216 5358
635 305 940


In [20]:
training_authors.to_json(path+'/train_authors.json')
test_authors.to_json(path+'/test_authors.json')

### Merge the writing style features of both phases

In [12]:
phase1_df = pd.read_csv('/mnt/swordfish-pool2/milad/hiatus-data/phase_1/explainability/filtered/refined_and_aggregated_features_final.csv')
feat_to_ling_lvl = json.load(open('/mnt/swordfish-pool2/milad/hiatus-data/phase_1/explainability/feats_to_ling_lvl.json'))
phase1_df['ling_lvl'] = phase1_df.original_attribute_name.apply(lambda x: feat_to_ling_lvl[x] if x in feat_to_ling_lvl else 'other')

phase2_df = pd.read_csv('/mnt/swordfish-pool2/milad/hiatus-data/phase_2/explainability/filtered/refined_and_aggregated_features_final.csv')
feat_to_ling_lvl = json.load(open('/mnt/swordfish-pool2/milad/hiatus-data/phase_2/explainability/feats_to_ling_lvl.json'))
phase2_df['ling_lvl'] = phase2_df.original_attribute_name.apply(lambda x: feat_to_ling_lvl[x] if x in feat_to_ling_lvl else 'other')

In [23]:
df = pd.concat([phase1_df, phase2_df]).reset_index()

In [26]:
print(df.documentID.nunique())
print(df['shortend_attribute_name.v2'].nunique(), df.aggregated_name.nunique(), df.final_attribute_name.nunique())
print(df.ling_lvl.nunique())

22706
9314 9049 2120
4


In [34]:
df.to_csv(path + '/refined_and_aggregated_features_final.csv', index=False)

In [36]:
df.groupby(['final_attribute_name', 'aggregated_name']).agg({'documentID': lambda x: len(x), 'ling_lvl': lambda x: list(x)[0]}).reset_index().to_csv(path + '/llm_generated_style_feats.csv')

In [28]:
g_df = df.groupby('final_attribute_name').agg({'original_attribute_name': lambda feats: {f: feats.tolist().count(f) for f in set(feats)},
                                               'shortend_attribute_name.v1': lambda feats: {f: feats.tolist().count(f) for f in set(feats)},
                                               'shortend_attribute_name.v2': lambda feats: {f: feats.tolist().count(f) for f in set(feats)},
                                               'aggregated_name': lambda feats: {f: feats.tolist().count(f) for f in set(feats)},
                                               'documentID': lambda x: len(x),
                                               'ling_lvl': lambda x: list(x)
                                    }).reset_index()

g_df['final_attribute_ling_lvl'] = g_df['ling_lvl'].apply(lambda x: Counter(x).most_common(1)[0][0])

In [30]:
g_df.to_json(path + '/style_features_corpus.json', orient='records', indent=2)

In [31]:
print(df.aggregated_name.nunique())
print(df.final_attribute_name.nunique())
g_df.final_attribute_ling_lvl.value_counts()

9049
2120


Semantic Level         673
Discourse Level        615
Syntactic Level        490
Morphological Level    342
Name: final_attribute_ling_lvl, dtype: int64

#### Loading the manually processed feats:

In [66]:
llm_based_df = pd.read_csv(path + '/refined_and_aggregated_features_final.csv')

In [67]:
cleaned_up_feats = pd.read_csv('/mnt/swordfish-pool2/milad/hiatus-data/explainability_all_data/manuall-processed-feats-nov-23-24.tsv', sep='\t')
cleaned_up_feats = cleaned_up_feats[cleaned_up_feats.documentID != 'documentID']
grouped_cleaned_up_feats_df = cleaned_up_feats.groupby('full_feature_processed').agg({
    'documentID': lambda x:sum([int(i) for i in x]),
    'full_feature_name': lambda x: list(x)
}).reset_index()

In [68]:
filtered_feats =  grouped_cleaned_up_feats_df[(grouped_cleaned_up_feats_df.documentID > 10)]
filtered_feats_map = {f: item[0] for item in zip(filtered_feats.full_feature_processed.tolist(), filtered_feats.full_feature_name.tolist()) for f in item[1]}

In [69]:
filtered_feats.sort_values('documentID', ascending=False)[['full_feature_processed', 'documentID']].head(n=5)

,full_feature_processed,documentID
2793,the author uses diverse sentence structures,11550
79,complex sentence structures are used,7066
3008,the author uses simple sentence structures,5880
232,specialized language is used,5337
214,sentence structures are varied,4740


In [70]:
llm_based_df['final_attribute_name_manually_processed'] = llm_based_df.aggregated_name.apply(lambda x: filtered_feats_map[x] if x in filtered_feats_map else '')
llm_based_df = llm_based_df[llm_based_df['final_attribute_name_manually_processed'] != '']

In [71]:
print('features: ', llm_based_df['final_attribute_name_manually_processed'].nunique())
print('documents: ',llm_based_df['documentID'].nunique())

features:  1377
documents:  22573


In [76]:
llm_based_df.to_csv(path + '/refined_and_aggregated_features_final_manually_processed_min_freq_10.csv', index=False)

#### Merge the Gram2vec style feats to our LLM-based style feats:

In [79]:
llm_based_df = pd.read_csv(path + '/refined_and_aggregated_features_final_manually_processed_min_freq_10.csv')
llm_based_df = llm_based_df.groupby('documentID').agg({'final_attribute_name_manually_processed': lambda x: list(x)}).reset_index()

In [80]:
gram2vec_df   = pd.read_json(path+ '/normalized_all_document_gram2vec_top_features.jsonl', lines=True)
gram2vec_dict = {x[0]: x[1] for x in zip(gram2vec_df.documentID.tolist(), gram2vec_df.gram2vec_feats.tolist())} 
gram2vec_feats = set([x for feats in gram2vec_dict.values() for x in feats])

In [81]:
llm_based_df['final_attribute_name_manually_processed'] = llm_based_df.apply(lambda x: x['final_attribute_name_manually_processed'] + gram2vec_dict[x['documentID']], axis=1)

In [82]:
llm_based_df = llm_based_df.explode(['final_attribute_name_manually_processed'])
gram2vec_df  = gram2vec_df.explode(['gram2vec_feats'])

In [83]:
llm_based_df.to_csv(path + '/llm_and_gram2vec_feats.csv', index=False)
gram2vec_df.to_csv(path+ '/gram2vec_feats.csv', index=False)

#### Annotated high/low level features:

Filter only style features that are low-level and verifiable

In [112]:
llm_based_df = pd.read_csv(path + '/refined_and_aggregated_features_final_manually_processed_min_freq_10.csv')
df = pd.read_csv(path +'/refined_and_aggregated_features_final_manually_processed_min_freq_10_filled.csv')
df['low_lvl_and_verifiable'] = df.apply(lambda row: row['Level'] == 'Low' and row['Verifiability'] == 'Yes', axis=1)
accepted_feats = df[df.Verifiability == 'Yes']['final_attribute_name_manually_processed'].tolist()

In [113]:
len(accepted_feats)

719

In [114]:
llm_based_df_filtered = llm_based_df[llm_based_df.final_attribute_name_manually_processed.isin(accepted_feats)]

In [115]:
llm_based_df_filtered.to_csv(path + '/refined_and_aggregated_features_final_manually_processed_w_low_and_verifiable_feats.csv')

In [116]:
df.low_lvl_and_verifiable.value_counts()

False    1063
True      314
Name: low_lvl_and_verifiable, dtype: int64

In [117]:
df.Verifiability.value_counts()

Yes    719
No     657
Name: Verifiability, dtype: int64

In [118]:
df.Level.value_counts()

High    1042
Low      335
Name: Level, dtype: int64

### Consturcted Interpretable space:

In [ ]:
! CUDA_VISIBLE_DEVICES=6 python ../cluster_documents.py --train-dir "/mnt/swordfish-pool2/milad/hiatus-data/explainability_all_data/train_authors.json" \
--test-dir "/mnt/swordfish-pool2/milad/hiatus-data/explainability_all_data/test_authors.json" \
--save-dir "/mnt/swordfish-pool2/milad/hiatus-data/explainability_all_data/" \
--model aa_model-luar \
--style-dir "/mnt/swordfish-pool2/milad/hiatus-data/explainability_all_data/refined_and_aggregated_features_final.csv"

Testing Different Epsilon Values:  88%|#######9 | 87/99 [44:09<05:51, 29.32s/it]

In [4]:
# This results in 44 clusters
! CUDA_VISIBLE_DEVICES=6 python ../cluster_documents.py --train-dir "/mnt/swordfish-pool2/milad/hiatus-data/explainability_all_data/train_authors.json" \
--test-dir "/mnt/swordfish-pool2/milad/hiatus-data/explainability_all_data/test_authors.json" \
--save-dir "/mnt/swordfish-pool2/milad/hiatus-data/explainability_all_data/" \
--model aa_model-luar \
--style-dir "/mnt/swordfish-pool2/milad/hiatus-data/explainability_all_data/refined_and_aggregated_features_final.csv" \
--style_feat_column 'final_attribute_name' \
--top_k_feats 10\
--summarize_cluster_reps\
--eps 0.07

# This results in 147 clusters
! yes| CUDA_VISIBLE_DEVICES=0 python ../cluster_documents.py --train-dir "/mnt/swordfish-pool2/milad/hiatus-data/explainability_all_data/train_authors.json" \
--test-dir "/mnt/swordfish-pool2/milad/hiatus-data/explainability_all_data/test_authors.json" \
--save-dir "/mnt/swordfish-pool2/milad/hiatus-data/explainability_all_data/interp_space_148_clusters/" \
--model  'aa_model-luar'\
--style-dir "/mnt/swordfish-pool2/milad/hiatus-data/explainability_all_data/refined_and_aggregated_features_final.csv" \
--style_feat_column 'final_attribute_name' \
--top_k_feats 10\
--eps 0.14
#--summarize_cluster_reps\

In [ ]:
# Running Clustering over LUAR -> resulting interpretable space of 39 clusters
! yes| CUDA_VISIBLE_DEVICES=0 python ../cluster_documents.py --train-dir "/mnt/swordfish-pool2/milad/hiatus-data/explainability_all_data/train_authors.json" --test-dir "/mnt/swordfish-pool2/milad/hiatus-data/explainability_all_data/test_authors.json" --save-dir "/mnt/swordfish-pool2/milad/hiatus-data/explainability_all_data/luar_interp_space_clusters/" --model  'luar-mud' --style-dir "/mnt/swordfish-pool2/milad/hiatus-data/explainability_all_data/refined_and_aggregated_features_final.csv" --style_feat_column 'final_attribute_name' --top_k_feats 10 --eps 0.06

In [16]:
# Running Clustering over our ta2 (Ajay's system trained on sadiri and then hrs)
! LORA_BASEMODEL_CHECKPOINT_PATH="/mnt/swordfish-pool2/milad/hiatus-data/models/original_ta2_performers_data_model_50k_hrs_continued_final_model/" CUDA_VISIBLE_DEVICES=0 python ../cluster_documents.py --train-dir "/mnt/swordfish-pool2/milad/hiatus-data/explainability_all_data/train_authors.json" --test-dir "/mnt/swordfish-pool2/milad/hiatus-data/explainability_all_data/test_authors.json" --save-dir "/mnt/swordfish-pool2/milad/hiatus-data/explainability_all_data/lora_ta2_interp_space_clusters/" --model  'luar-mud' --style-dir "/mnt/swordfish-pool2/milad/hiatus-data/explainability_all_data/refined_and_aggregated_features_final.csv" --style_feat_column 'final_attribute_name' --top_k_feats 10 --eps 0.16

huggingface/tokenizers: The current process just got forked, after parallelism has already been used. Disabling parallelism to avoid deadlocks...
To disable this warning, you can either:
	- Avoid using `tokenizers` before the fork if possible
	- Explicitly set the environment variable TOKENIZERS_PARALLELISM=(true | false)


base model /mnt/swordfish-pool2/milad/hiatus-data/models/original_ta2_performers_data_model_50k_hrs_continued_final_model/


In [15]:
# Running Clustering over our ta2 (Ajay's system trained on hrs)
# EPS = 0.09 -> [(0.094, 0.163, 0.378, 0.919), 56, 0.17665369318776145, 0] 0.09
! LORA_BASEMODEL_CHECKPOINT_PATH="/mnt/swordfish-pool2/milad/hiatus-data/models/ta2-system2-en-submitted-8-01-25/rrivera1849/LUAR-MUD/" LORA_ADAPTER_CHECKPOINT_PATH="/mnt/swordfish-pool2/milad/hiatus-data/models/ta2-system2-en-submitted-8-01-25/lora_checkpoint/" CUDA_VISIBLE_DEVICES=0 python cluster_documents.py --train-dir "/mnt/swordfish-pool2/milad/hiatus-data/explainability_all_data/train_authors.json" --test-dir "/mnt/swordfish-pool2/milad/hiatus-data/explainability_all_data/test_authors.json" --save-dir "/mnt/swordfish-pool2/milad/hiatus-data/explainability_all_data/system2_interp_space_clusters/" --model  'datadreamer-lora' --style-dir "/mnt/swordfish-pool2/milad/hiatus-data/explainability_all_data/refined_and_aggregated_features_final.csv" --style_feat_column 'final_attribute_name' --top_k_feats 10 --eps 0.09

### Build Clusters Style Representation

In [8]:
#llm_style_feats = path + '/refined_and_aggregated_features_final.csv'
#llm_style_feats = path + '/refined_and_aggregated_features_final_manually_processed.csv'
llm_style_feats = path + 'refined_and_aggregated_features_final_manually_processed_w_low_and_verifiable_feats.csv'
llm_style_feats_dict = path + '/refined_and_aggregated_features_final_manually_processed_min_freq_10_filled.csv'
llm_and_g2v_style_feats = path + '/llm_and_gram2vec_feats.csv'
g2v_style_feats = path + '/gram2vec_feats.csv'

In [120]:
# df = pd.read_csv(llm_style_feats)
# df[['shortend_attribute_name.v2', 'original_attribute_name']].drop_duplicates().to_csv('/mnt/swordfish-pool2/milad/hiatus-data/explainability_all_data/original_feature_name.csv')

In [9]:
def build_cluster_representation(clustering_path, output_path, top_k=10, summarize_with_gpt=False):
    #feat_clm = 'final_attribute_name'
    styles_corpus_path = llm_style_feats
    feat_clm = 'final_attribute_name_manually_processed'

    # We are not using the featus_dict for now
    df = pd.read_csv(llm_style_feats_dict)
    feats_dict = {x[0]: (x[1], x[2]) for x in zip(df[feat_clm].tolist(), df.Level.tolist(), df.Verifiability.tolist())}
    
    #Representative Summarization
    clusters_tfidf_rep_df = generate_interpretable_space_representation(clustering_path, styles_corpus_path, feat_clm, 'tfidf_rep', num_feats=top_k, summarize_with_gpt=summarize_with_gpt)
    #Contrastive Summarization
    #clusters_contra_rep_df = generate_interpretable_space_contra_representation(clustering_path, styles_corpus_path, feat_clm, 'con_rep', num_feats=top_k, summarize_with_gpt=summarize_with_gpt)

    feat_clm = 'gram2vec_feats'
    styles_corpus_path = g2v_style_feats
    #Representative Summarization
    clusters_tfidf_rep_g2v_df = generate_interpretable_space_representation(clustering_path, styles_corpus_path, feat_clm, 'tfidf_rep', num_feats=top_k, summarize_with_gpt=summarize_with_gpt)
    #Contrastive Summarization
    #clusters_contra_rep_g2v_df = generate_interpretable_space_contra_representation(clustering_path, styles_corpus_path, feat_clm, 'con_rep', num_feats=top_k, summarize_with_gpt=summarize_with_gpt)

    #clusters_tfidf_rep_df['llm_con_rep'] = clusters_contra_rep_df['con_rep']
    clusters_tfidf_rep_df['llm_tfidf_rep'] = clusters_tfidf_rep_df['tfidf_rep']
    clusters_tfidf_rep_df['llm_tfidf_weights'] = clusters_tfidf_rep_df['tfidf_rep_dist']
    clusters_tfidf_rep_df['g2v_tfidf_rep'] = clusters_tfidf_rep_g2v_df['tfidf_rep']
    clusters_tfidf_rep_df['g2v_tfidf_weights'] = clusters_tfidf_rep_g2v_df['tfidf_rep_dist']
    #clusters_tfidf_rep_df['g2v_con_rep'] = clusters_contra_rep_g2v_df['con_rep']

    clusters_tfidf_rep_df[['cluster_label', 'llm_tfidf_rep', 'llm_tfidf_weights', 'g2v_tfidf_rep', 'g2v_tfidf_weights']].to_json(output_path)
    
    return clusters_tfidf_rep_df

In [24]:
build_cluster_representation('/mnt/swordfish-pool2/milad/hiatus-data/explainability_all_data/train_authors.pkl', 
                             '/mnt/swordfish-pool2/milad/hiatus-data/explainability_all_data/interpretable_space_representations.json', top_k=10)

In [30]:
build_cluster_representation('/mnt/swordfish-pool2/milad/hiatus-data/explainability_all_data/interp_space_148_clusters/train_authors.pkl', 
                             '/mnt/swordfish-pool2/milad/hiatus-data/explainability_all_data/interp_space_148_clusters/interpretable_space_representations.json', top_k=10)

In [122]:
resulted_df = build_cluster_representation('/mnt/swordfish-pool2/milad/hiatus-data/explainability_all_data/lora_ta2_interp_space_clusters/train_authors.pkl', 
                             '/mnt/swordfish-pool2/milad/hiatus-data/explainability_all_data/lora_ta2_interp_space_clusters/interpretable_space_representations.json', top_k=10, summarize_with_gpt=True)

Number of style feats  719


[ 🤖 DataDreamer 💤 ] Initialized. 🚀 Dreaming to folder: .datadreamer/summarize/openai:gpt-3.5-turbo/summarize_sentences


The token has not been saved to the git credentials helper. Pass `add_to_git_credential=True` in this function directly or `--add-to-git-credential` if using via `huggingface-cli` if you want to set the git credential as well.
Token is valid (permission: read).
Your token has been saved to /home/ma4608/.cache/huggingface/token
Login successful
Summarizing styles of interpretable dimensions


[ 🤖 DataDreamer 💤 ] Step 'sentences' was previously run and saved, but was outdated. 😞
[ 🤖 DataDreamer 💤 ] Step 'sentences' is running. ⏳
[ 🤖 DataDreamer 💤 ] Step 'sentences' finished and is saved to disk. 🎉
[ 🤖 DataDreamer 💤 ] Step 'Summarize Sentences' was previously run and saved, but was outdated. 😞
[ 🤖 DataDreamer 💤 ] Step 'Summarize Sentences' is running. ⏳
[ 🤖 DataDreamer 💤 ] Step 'Summarize Sentences' finished and is saved to disk. 🎉
[ 🤖 DataDreamer 💤 ] Step 'Summarize Sentences (select_columns)' is running. ⏳
[ 🤖 DataDreamer 💤 ] Step 'Summarize Sentences (select_columns)' finished running lazily. 🎉
[ 🤖 DataDreamer 💤 ] Step 'zipped(sentences, Summarize Sentences (select_columns))' is running. ⏳
[ 🤖 DataDreamer 💤 ] Step 'zipped(sentences, Summarize Sentences (select_columns))' will run lazily. 🥱
[ 🤖 DataDreamer 💤 ] Step 'zipped(sentences, Summarize Sentences (select_columns))' finished running lazily. 🎉
[ 🤖 DataDreamer 💤 ] Done. ✨ Results in folder: .datadreamer/summarize/openai

Number of style feats  448


[ 🤖 DataDreamer 💤 ] Initialized. 🚀 Dreaming to folder: .datadreamer/summarize/openai:gpt-3.5-turbo/summarize_sentences
[ 🤖 DataDreamer 💤 ] Step 'sentences' was previously run and saved, but was outdated. 😞
[ 🤖 DataDreamer 💤 ] Step 'sentences' was previously run and the results were backed up. 💾


The token has not been saved to the git credentials helper. Pass `add_to_git_credential=True` in this function directly or `--add-to-git-credential` if using via `huggingface-cli` if you want to set the git credential as well.
Token is valid (permission: read).
Your token has been saved to /home/ma4608/.cache/huggingface/token
Login successful
Summarizing styles of interpretable dimensions


[ 🤖 DataDreamer 💤 ] Step 'sentences' results loaded from disk. 🙌 It was previously run and saved.
[ 🤖 DataDreamer 💤 ] Step 'Summarize Sentences' was previously run and saved, but was outdated. 😞
[ 🤖 DataDreamer 💤 ] Step 'Summarize Sentences' was previously run and the results were backed up. 💾
[ 🤖 DataDreamer 💤 ] Step 'Summarize Sentences' results loaded from disk. 🙌 It was previously run and saved.
[ 🤖 DataDreamer 💤 ] Step 'Summarize Sentences (select_columns)' is running. ⏳
[ 🤖 DataDreamer 💤 ] Step 'Summarize Sentences (select_columns)' finished running lazily. 🎉
[ 🤖 DataDreamer 💤 ] Step 'zipped(sentences, Summarize Sentences (select_columns))' is running. ⏳
[ 🤖 DataDreamer 💤 ] Step 'zipped(sentences, Summarize Sentences (select_columns))' will run lazily. 🥱
[ 🤖 DataDreamer 💤 ] Step 'zipped(sentences, Summarize Sentences (select_columns))' finished running lazily. 🎉
[ 🤖 DataDreamer 💤 ] Done. ✨ Results in folder: .datadreamer/summarize/openai:gpt-3.5-turbo/summarize_sentences


In [10]:
resulted_df = build_cluster_representation('/mnt/swordfish-pool2/milad/hiatus-data/explainability_all_data/system2_interp_space_clusters/train_authors.pkl', 
                             '/mnt/swordfish-pool2/milad/hiatus-data/explainability_all_data/system2_interp_space_clusters/interpretable_space_representations.json', top_k=10, summarize_with_gpt=True)

Number of style feats  719


[ 🤖 DataDreamer 💤 ] Initialized. 🚀 Dreaming to folder: .datadreamer/summarize/openai:gpt-3.5-turbo/summarize_sentences
[ 🤖 DataDreamer 💤 ] Step 'sentences' was previously run and saved, but was outdated. 😞


The token has not been saved to the git credentials helper. Pass `add_to_git_credential=True` in this function directly or `--add-to-git-credential` if using via `huggingface-cli` if you want to set the git credential as well.
Token is valid (permission: read).
Your token has been saved to /home/ma4608/.cache/huggingface/token
Login successful
Summarizing styles of interpretable dimensions


[ 🤖 DataDreamer 💤 ] Step 'sentences' is running. ⏳
[ 🤖 DataDreamer 💤 ] Step 'sentences' finished and is saved to disk. 🎉
[ 🤖 DataDreamer 💤 ] Step 'Summarize Sentences' was previously run and saved, but was outdated. 😞
[ 🤖 DataDreamer 💤 ] Step 'Summarize Sentences' is running. ⏳
[ 🤖 DataDreamer 💤 ] Step 'Summarize Sentences' finished and is saved to disk. 🎉
[ 🤖 DataDreamer 💤 ] Step 'Summarize Sentences (select_columns)' is running. ⏳
[ 🤖 DataDreamer 💤 ] Step 'Summarize Sentences (select_columns)' finished running lazily. 🎉
[ 🤖 DataDreamer 💤 ] Step 'zipped(sentences, Summarize Sentences (select_columns))' is running. ⏳
[ 🤖 DataDreamer 💤 ] Step 'zipped(sentences, Summarize Sentences (select_columns))' will run lazily. 🥱
[ 🤖 DataDreamer 💤 ] Step 'zipped(sentences, Summarize Sentences (select_columns))' finished running lazily. 🎉
[ 🤖 DataDreamer 💤 ] Done. ✨ Results in folder: .datadreamer/summarize/openai:gpt-3.5-turbo/summarize_sentences


Number of style feats  448


[ 🤖 DataDreamer 💤 ] Initialized. 🚀 Dreaming to folder: .datadreamer/summarize/openai:gpt-3.5-turbo/summarize_sentences
[ 🤖 DataDreamer 💤 ] Step 'sentences' was previously run and saved, but was outdated. 😞
[ 🤖 DataDreamer 💤 ] Step 'sentences' is running. ⏳


The token has not been saved to the git credentials helper. Pass `add_to_git_credential=True` in this function directly or `--add-to-git-credential` if using via `huggingface-cli` if you want to set the git credential as well.
Token is valid (permission: read).
Your token has been saved to /home/ma4608/.cache/huggingface/token
Login successful
Summarizing styles of interpretable dimensions


[ 🤖 DataDreamer 💤 ] Step 'sentences' finished and is saved to disk. 🎉
[ 🤖 DataDreamer 💤 ] Step 'Summarize Sentences' was previously run and saved, but was outdated. 😞
[ 🤖 DataDreamer 💤 ] Step 'Summarize Sentences' is running. ⏳
[ 🤖 DataDreamer 💤 ] Step 'Summarize Sentences' finished and is saved to disk. 🎉
[ 🤖 DataDreamer 💤 ] Step 'Summarize Sentences (select_columns)' is running. ⏳
[ 🤖 DataDreamer 💤 ] Step 'Summarize Sentences (select_columns)' finished running lazily. 🎉
[ 🤖 DataDreamer 💤 ] Step 'zipped(sentences, Summarize Sentences (select_columns))' is running. ⏳
[ 🤖 DataDreamer 💤 ] Step 'zipped(sentences, Summarize Sentences (select_columns))' will run lazily. 🥱
[ 🤖 DataDreamer 💤 ] Step 'zipped(sentences, Summarize Sentences (select_columns))' finished running lazily. 🎉
[ 🤖 DataDreamer 💤 ] Done. ✨ Results in folder: .datadreamer/summarize/openai:gpt-3.5-turbo/summarize_sentences


In [147]:
# Converting gra2vec style corpus from jsonl to csv
# g2v_style_corpus = pd.read_json('/mnt/swordfish-pool2/milad/hiatus-data/explainability_all_data/normalized_all_document_gram2vec_top_features.jsonl', lines=True)
# g2v_style_corpus = g2v_style_corpus.explode('gram2vec_feats')
# g2v_style_corpus.to_csv('/mnt/swordfish-pool2/milad/hiatus-data/explainability_all_data/normalized_all_document_gram2vec_top_features.csv', index=False)